# Exact Rational Certificate for Condition (a)

This notebook gives an exact rational verification of condition (a) in Theorem 5 for the candidate
$$(\lambda^\ast,s_0^\ast,s_1^\ast,s_2^\ast)$$
selected in Section 4.2.

The goal is to prove the stronger sufficient estimate
$$
|s_0^\ast|+|s_1^\ast|+|s_2^\ast|<1.
$$
By Rouché's theorem, this implies that all zeros of
$$
h(z)=z^3+s_2^\ast z^2+s_1^\ast z+s_0^\ast
$$
lie in the open unit disk.

All arithmetic below is exact over $\mathbb Q$ using Python's `fractions.Fraction`.
No floating-point arithmetic is used in any certificate check.


## Mathematical setup

The RealRootCounting step establishes that the selected candidate $s_0^\ast$ lies in the rectangle
$$
I=\left\{x+iy:\ 
x\in\left[\frac{160734}{10^9},\frac{160735}{10^9}\right],
\quad
y\in\left[-\frac{6419166}{10^9},-\frac{6419165}{10^9}\right]\right\}.
$$

We use the rational reference point
$$
\widehat s_0=\frac{160734}{10^9}-\frac{6419165}{10^9}i.
$$
For every $s\in I$,
$$
|s-\widehat s_0|\le \sqrt 2\cdot 10^{-9}<2\cdot 10^{-9}=:\delta.
$$
Moreover, both $s$ and $\widehat s_0$ satisfy
$$
|s|\le R,\qquad |\widehat s_0|\le R,\qquad R:=\frac{1}{100}.
$$

Using the plus-branch relation and eliminating the radical gives
$$
s_1^\ast=\frac{f_1(s_0^\ast)f_5(s_0^\ast)-f_4(s_0^\ast)}{4f_5(s_0^\ast)}
$$
and
$$
s_2^\ast=\frac{f_4(s_0^\ast)-f_2(s_0^\ast)f_5(s_0^\ast)}
{4s_0^\ast f_5(s_0^\ast)}.
$$
Define
$$
N_1(z):=f_1(z)f_5(z)-f_4(z),\qquad
N_2(z):=f_4(z)-f_2(z)f_5(z).
$$

It is enough to prove the following exact rational bounds:
$$
|s_0^\ast|<\frac1{100},\qquad
|N_1(s_0^\ast)|<\frac{47}{5},\qquad
|N_2(s_0^\ast)|<\frac{597}{1000},
$$
and
$$
|s_0^\ast|>\frac{321}{50000},\qquad
|f_5(s_0^\ast)|>\frac{1349}{50}.
$$
Indeed, these imply
$$
|s_1^\ast|
<
\frac{47/5}{4(1349/50)}
<\frac9{100}
$$
and
$$
|s_2^\ast|
<
\frac{597/1000}{4(321/50000)(1349/50)}
<\frac{87}{100}.
$$
Therefore
$$
|s_0^\ast|+|s_1^\ast|+|s_2^\ast|
<
\frac1{100}+\frac9{100}+\frac{87}{100}
=
\frac{97}{100}
<1.
$$


In [1]:
from fractions import Fraction

# -----------------------------------------------------------------------------
# Complex rational arithmetic, represented as pairs (real, imaginary)
# -----------------------------------------------------------------------------

def cadd(a, b):
    return (a[0] + b[0], a[1] + b[1])

def csub(a, b):
    return (a[0] - b[0], a[1] - b[1])

def cmul(a, b):
    return (a[0]*b[0] - a[1]*b[1], a[0]*b[1] + a[1]*b[0])

def cabs2(a):
    return a[0]*a[0] + a[1]*a[1]

def poly_eval(coeffs, z):
    # Evaluate p(z)=sum_k coeffs[k] z^k exactly.
    acc = (Fraction(0), Fraction(0))
    for c in reversed(coeffs):
        acc = cmul(acc, z)
        acc = (acc[0] + c, acc[1])
    return acc

def poly_add(p, q):
    n = max(len(p), len(q))
    out = [Fraction(0) for _ in range(n)]
    for i in range(n):
        out[i] = (p[i] if i < len(p) else Fraction(0)) + (q[i] if i < len(q) else Fraction(0))
    return out

def poly_sub(p, q):
    n = max(len(p), len(q))
    out = [Fraction(0) for _ in range(n)]
    for i in range(n):
        out[i] = (p[i] if i < len(p) else Fraction(0)) - (q[i] if i < len(q) else Fraction(0))
    return out

def poly_mul(p, q):
    out = [Fraction(0) for _ in range(len(p) + len(q) - 1)]
    for i, a in enumerate(p):
        for j, b in enumerate(q):
            out[i+j] += a*b
    return out

def derivative_bound(coeffs, R):
    # Return sum_{k>=1} |a_k| k R^{k-1}.
    # If |z|,|w| <= R, then
    # |p(z)-p(w)| <= |z-w| * derivative_bound(coeffs,R).
    total = Fraction(0)
    for k, a in enumerate(coeffs):
        if k >= 1 and a != 0:
            total += abs(a) * k * (R ** (k-1))
    return total

def require(name, condition):
    if not condition:
        raise AssertionError(f"FAILED: {name}")
    print(f"PASS: {name}")


## Define the polynomials

The polynomials are stored in ascending coefficient order:
$$
p(z)=\sum_{k=0}^d a_k z^k.
$$


In [2]:
# f1(z)=2z^2-27z-2
f1 = [Fraction(-2), Fraction(-27), Fraction(2)]

# f2(z)=2z^2+27z-2
f2 = [Fraction(-2), Fraction(27), Fraction(2)]

# f4 and f5 from the paper, ascending coefficient order
f4 = [Fraction(0)] * 17
for k, v in {
    16:-8192, 15:499712, 14:-7077888, 13:-31111168,
    12:659681792, 11:-326163456, 10:-5979808768,
    9:7396888576, 8:3606093312, 7:-22677711616,
    6:2524591872, 5:1298253312, 4:-54335488,
    3:-8839168, 2:552960, 1:-8192
}.items():
    f4[k] = Fraction(v)

f5 = [Fraction(0)] * 15
for k, v in {
    14:-4096, 13:194560, 12:-2022400, 11:-5172736,
    10:72910848, 9:-65459200, 8:-205725696,
    7:369364992, 6:-284532480, 5:-194358528,
    4:16606208, 3:2543616, 2:-221184, 1:4096
}.items():
    f5[k] = Fraction(v)

# N1(z)=f1(z)f5(z)-f4(z)
# N2(z)=f4(z)-f2(z)f5(z)
N1 = poly_sub(poly_mul(f1, f5), f4)
N2 = poly_sub(f4, poly_mul(f2, f5))

print("degree N1 =", len(N1)-1)
print("degree N2 =", len(N2)-1)


degree N1 = 16
degree N2 = 16


## Exact rational neighborhood data

We use
$$
\delta=\frac{2}{10^9},\qquad R=\frac1{100},
$$
so that $|s_0^\ast-\widehat s_0|\le\delta$ and all points under consideration lie in $|z|\le R$.


In [3]:
hat_s0 = (Fraction(160734, 10**9), -Fraction(6419165, 10**9))
delta = Fraction(2, 10**9)
R = Fraction(1, 100)

# Rectangle endpoints
xmin = Fraction(160734, 10**9)
xmax = Fraction(160735, 10**9)
ymin = -Fraction(6419166, 10**9)
ymax = -Fraction(6419165, 10**9)

# Candidate rational constants used in the certificate
U0 = Fraction(1, 100)
L0 = Fraction(321, 50000)
L5 = Fraction(1349, 50)
UN1 = Fraction(47, 5)
UN2 = Fraction(597, 1000)
U1 = Fraction(9, 100)
U2 = Fraction(87, 100)


## Bounds for $|s_0^\ast|$

For every $s=x+iy\in I$,
$$
|s|^2=x^2+y^2.
$$
The upper bound is obtained from the largest endpoint magnitudes, and the lower bound from the smallest endpoint magnitudes:
$$
|s_0^\ast|<\frac1{100},\qquad
|s_0^\ast|>\frac{321}{50000}.
$$
Both are verified by comparing squares, hence no square roots are used.


In [4]:
# Upper bound: max |s| over the rectangle is bounded by xmax^2 + |ymin|^2
require(
    "|s0*| < 1/100",
    xmax*xmax + abs(ymin)*abs(ymin) < U0*U0
)

# Lower bound: min |s| over the rectangle is bounded below by xmin^2 + |ymax|^2
require(
    "|s0*| > 321/50000",
    L0*L0 < xmin*xmin + abs(ymax)*abs(ymax)
)


PASS: |s0*| < 1/100
PASS: |s0*| > 321/50000


## Polynomial perturbation bounds

For any polynomial
$$
p(z)=\sum_{k=0}^d a_k z^k
$$
and any $z,w$ satisfying $|z|,|w|\le R$, we have
$$
|p(z)-p(w)|
\le
|z-w|
\sum_{k=1}^d |a_k|\,k\,R^{k-1}.
$$
This follows from
$$
z^k-w^k=(z-w)\sum_{j=0}^{k-1} z^{k-1-j}w^j.
$$

We apply this estimate with $z=s_0^\ast$ and $w=\widehat s_0$.


In [5]:
def error_bound(coeffs):
    return delta * derivative_bound(coeffs, R)

F5 = poly_eval(f5, hat_s0)
FN1 = poly_eval(N1, hat_s0)
FN2 = poly_eval(N2, hat_s0)

E5 = error_bound(f5)
EN1 = error_bound(N1)
EN2 = error_bound(N2)

print("E5  =", E5)
print("EN1 =", EN1)
print("EN2 =", EN2)


E5  = 57123296386478531737734191/3051757812500000000000000000000
EN1 = 2320904019720202818669817/152587890625000000000000000000
EN2 = 255905516139352104101018081/122070312500000000000000000000000


## Lower bound for $|f_5(s_0^\ast)|$

We prove
$$
|f_5(s_0^\ast)|>\frac{1349}{50}.
$$
Since
$$
|f_5(s_0^\ast)-f_5(\widehat s_0)|\le E_5,
$$
it is enough to prove
$$
|f_5(\widehat s_0)|>\frac{1349}{50}+E_5.
$$
This is checked exactly by comparing squares:
$$
\left(\frac{1349}{50}+E_5\right)^2
<
|f_5(\widehat s_0)|^2.
$$


In [6]:
require(
    "|f5(s0*)| > 1349/50",
    (L5 + E5)*(L5 + E5) < cabs2(F5)
)


PASS: |f5(s0*)| > 1349/50


## Upper bounds for $|N_1(s_0^\ast)|$ and $|N_2(s_0^\ast)|$

We prove
$$
|N_1(s_0^\ast)|<\frac{47}{5},
\qquad
|N_2(s_0^\ast)|<\frac{597}{1000}.
$$

For example, since
$$
|N_1(s_0^\ast)-N_1(\widehat s_0)|\le E_{N_1},
$$
it suffices to check
$$
|N_1(\widehat s_0)|<\frac{47}{5}-E_{N_1}.
$$
Again this is verified by comparing squares.


In [7]:
require(
    "|N1(s0*)| < 47/5",
    (UN1 - EN1) > 0 and cabs2(FN1) < (UN1 - EN1)*(UN1 - EN1)
)

require(
    "|N2(s0*)| < 597/1000",
    (UN2 - EN2) > 0 and cabs2(FN2) < (UN2 - EN2)*(UN2 - EN2)
)


PASS: |N1(s0*)| < 47/5
PASS: |N2(s0*)| < 597/1000


## Consequences for $s_1^\ast$ and $s_2^\ast$

From
$$
s_1^\ast=\frac{N_1(s_0^\ast)}{4f_5(s_0^\ast)}
$$
we get
$$
|s_1^\ast|
<
\frac{47/5}{4(1349/50)}
<
\frac9{100}.
$$

From
$$
s_2^\ast=\frac{N_2(s_0^\ast)}{4s_0^\ast f_5(s_0^\ast)}
$$
we get
$$
|s_2^\ast|
<
\frac{597/1000}{4(321/50000)(1349/50)}
<
\frac{87}{100}.
$$


In [8]:
require(
    "|s1*| < 9/100",
    UN1 / (4 * L5) < U1
)

require(
    "|s2*| < 87/100",
    UN2 / (4 * L0 * L5) < U2
)


PASS: |s1*| < 9/100
PASS: |s2*| < 87/100


## Final certificate for condition (a)

Combining the bounds,
$$
|s_0^\ast|+|s_1^\ast|+|s_2^\ast|
<
\frac1{100}+\frac9{100}+\frac{87}{100}
=
\frac{97}{100}
<1.
$$
This verifies condition (a) by Rouché's theorem.


In [9]:
require(
    "|s0*| + |s1*| + |s2*| < 1",
    U0 + U1 + U2 < 1
)

print("Exact rational certificate for condition (a) completed successfully.")


PASS: |s0*| + |s1*| + |s2*| < 1
Exact rational certificate for condition (a) completed successfully.


## Suggested lemma statement for the paper

The notebook verifies the following lemma.

```latex
\begin{lemma}
\label{lemma:condition_a_verification}
For the candidate quadruple
\((\Sollda,\Sols_0,\Sols_1,\Sols_2)\) obtained from the plus-sign branch,
one has
\[
    |\Sols_0|<\frac1{100},\qquad
    |\Sols_1|<\frac9{100},\qquad
    |\Sols_2|<\frac{87}{100}.
\]
Consequently,
\[
    |\Sols_0|+|\Sols_1|+|\Sols_2|<1.
\]
In particular, all zeros of
\[
    h(z)=z^3+\Sols_2 z^2+\Sols_1 z+\Sols_0
\]
lie in the open unit disk.
\end{lemma}

\begin{proof}
The proof is given by the exact rational certificate in the supplementary
notebook. All computations are carried out over \(\mathbb Q\).
\end{proof}
```
